# 第六轮：频带时长神经群
逐帧输入、逐帧更新。输入侧增加多时间尺度持续刺激痕迹与新刺激响应；输出侧增加持续活动与适应。模型运行时不读取显式段长、倒计时或原始输入历史。先运行冒烟测试，再正式训练。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码版本:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/Parrot')
result_dir = drive_root / 'trial6_results'
result_dir.mkdir(parents=True, exist_ok=True)
print(result_dir)


## 快速检查
确认模型、反向传播及文件写入能运行。


In [ ]:
!python trial6.py --smoke --batch 3 --out /content/Parrot/trial6_smoke


## 正式训练与评估
默认 3000 步，与第五轮保持相同数据生成和评估种子。训练日志每 100 步保存一次。Colab 中断时，请保留 Drive 内已有结果并回传异常信息。


In [ ]:
!python trial6.py --steps 3000 --batch 20 --eval 300 --out {result_dir}


In [ ]:
import json
from IPython.display import Audio, display
report_path = result_dir / 'metrics.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
report['git_commit'] = commit
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
for gain, groups in report['conditions'].items():
    print('回传增益:', gain, '总体:', groups['overall'], '重复:', groups['repeated'])
display(Audio(filename=str(result_dir / 'input.wav')))
display(Audio(filename=str(result_dir / 'model.wav')))


## 回传
运行下方单元，将生成的 ZIP 下载或放入项目的 `result/` 目录。


In [ ]:
import shutil
archive = shutil.make_archive(str(drive_root / 'trial6_results'), 'zip', root_dir=str(drive_root), base_dir='trial6_results')
print('回传:', archive)
